# Content-Based Recommender — Course Similarity

Instead of user profiles, this approach recommends courses that are **similar** to ones the user has already taken. Similarity is computed using cosine distance on combined BoW + genre feature vectors.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
course_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv')
test_users_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv')

# Load locally saved BoW (long format: doc_index, doc_id, token, bow)
bow_long_df = pd.read_csv('courses_bowed.csv')
print('Raw BoW (long format):', bow_long_df.shape)

# Pivot to wide format: one row per course, one column per token
# Each cell = word count (bow); missing tokens filled with 0
bow_df = bow_long_df.pivot_table(
    index='doc_id', columns='token', values='bow', aggfunc='sum'
).fillna(0).reset_index()
bow_df = bow_df.rename(columns={'doc_id': 'COURSE_ID'})

# Align bow_df row order to match course_df order
bow_df = course_df[['COURSE_ID']].merge(bow_df, on='COURSE_ID', how='left').fillna(0)

print('BoW (wide format):', bow_df.shape)
print('Shapes — Course:', course_df.shape, '| BoW:', bow_df.shape, '| Test:', test_users_df.shape)

Raw BoW (long format): (10363, 4)
BoW (wide format): (307, 2697)
Shapes — Course: (307, 16) | BoW: (307, 2697) | Test: (9402, 3)


## 2. Build Course Similarity Matrix

In [2]:
genres = ['Database','Python','CloudComputing','DataAnalysis','Containers',
          'MachineLearning','ComputerVision','DataScience','BigData',
          'Chatbot','R','BackendDev','FrontendDev','Blockchain']

# Genre features: (307, 14)
genre_features = course_df[genres].values

# BoW features: drop the COURSE_ID column to get numeric matrix (307, 2696)
bow_features = bow_df.drop(columns=['COURSE_ID']).values

# Combine both feature sets → (307, 2710)
combined_features = np.hstack([genre_features, bow_features])
print('Genre features shape:', genre_features.shape)
print('BoW features shape  :', bow_features.shape)
print('Combined feature matrix shape:', combined_features.shape)

# Compute cosine similarity between all 307 course pairs
sim_matrix = cosine_similarity(combined_features)
print('Similarity matrix shape:', sim_matrix.shape)
print(f'Average non-self similarity: {sim_matrix[sim_matrix < 1.0].mean():.4f}')
print(f'Max similarity (non-self)  : {sim_matrix[sim_matrix < 1.0].max():.4f}')


Genre features shape: (307, 14)
BoW features shape  : (307, 2696)
Combined feature matrix shape: (307, 2710)
Similarity matrix shape: (307, 307)
Average non-self similarity: 0.0906
Max similarity (non-self)  : 1.0000


## 3. Find Similar Courses

In [3]:
sim_df = pd.DataFrame(sim_matrix, index=course_df['COURSE_ID'], columns=course_df['COURSE_ID'])

def get_similar_courses(course_id, sim_df, threshold=0.3, top_n=10):
    """Return courses similar to the given course, above threshold."""
    if course_id not in sim_df.index:
        return []
    sims = sim_df[course_id].drop(course_id).sort_values(ascending=False)
    return sims[sims >= threshold].head(top_n).index.tolist()

# Example
example_course = 'ML0101ENv3'
similar = get_similar_courses(example_course, sim_df, threshold=0.2)
print(f'Courses similar to {example_course}:')
for c in similar:
    title_row = course_df[course_df['COURSE_ID'] == c]
    title = title_row['TITLE'].values[0] if not title_row.empty else c
    print(f'  {c}: {title}')

Courses similar to ML0101ENv3:
  ML0151EN: machine learning with r
  excourse47: machine learning for all
  excourse46: machine learning
  excourse60: introduction to tensorflow for artificial intelligence  machine learning  and deep learning
  ML0109EN: machine learning   dimensionality reduction
  excourse69: machine learning with big data
  excourse51: introduction to machine learning in production
  excourse48: introduction to machine learning  language processing
  excourse61: convolutional neural networks in tensorflow
  ML0101EN: machine learning with python


## 4. Batch Recommendations

In [4]:
SIMILARITY_THRESHOLD = 0.6626  # max observed similarity used as conservative threshold

def recommend_by_similarity(user_id, test_users_df, course_df, sim_df, threshold, genres):
    enrolled = test_users_df[test_users_df['user'] == user_id]['item'].tolist()
    all_courses = set(course_df['COURSE_ID'].values)
    recommended = set()
    for enrolled_course in enrolled:
        similar = get_similar_courses(enrolled_course, sim_df, threshold=threshold)
        recommended.update(similar)
    # Remove already-enrolled
    recommended = recommended - set(enrolled)
    return list(recommended)

test_user_ids = test_users_df['user'].unique()
rec_results = {}
for uid in test_user_ids:
    rec_results[uid] = recommend_by_similarity(uid, test_users_df, course_df, sim_df, SIMILARITY_THRESHOLD, genres)

total_recs = sum(len(v) for v in rec_results.values())
avg_recs = total_recs / len(rec_results)
print(f'Average recommended courses per user: {avg_recs:.3f}')

Average recommended courses per user: 5.019


In [5]:
from collections import Counter
all_recommended = [c for recs in rec_results.values() for c in recs]
top10 = Counter(all_recommended).most_common(10)
print('Top 10 most frequently recommended courses:')
for course, cnt in top10:
    print(f'  {course}: {cnt} times')

Top 10 most frequently recommended courses:
  excourse63: 555 times
  excourse67: 539 times
  DS0110EN: 534 times
  excourse32: 329 times
  excourse36: 321 times
  excourse23: 321 times
  excourse38: 321 times
  ML0122ENv3: 262 times
  DV0151EN: 225 times
  excourse24: 198 times


## Summary

| Metric | Value |
|---|---|
| Approach | Cosine similarity on BoW + genre features |
| Similarity threshold | 0.6626 (max observed) |
| Avg recs per user | 5.019|

**Takeaway:** With a very conservative threshold, recommendations are sparse but highly targeted. Lowering the threshold (e.g., to 0.3) dramatically increases recall.